# Lab 01: LLM API Foundation - Interactive Test & Verification
Notebook này cho phép bạn chạy thử nghiệm và kiểm tra trực quan kết quả của từng Task trong bài thực hành.

In [ ]:
import os
import sys
from pathlib import Path
from unittest.mock import MagicMock, patch

# Thiết lập đường dẫn import
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Import các hàm từ solution
from solution.solution import (
    call_openai,
    call_gemini,
    call_anthropic,
    compare_models,
    retry_with_backoff,
    batch_compare,
    format_comparison_table,
    PRICING_1M_TOKENS
)

print("✅ Import thành công các hàm từ solution.py!")

--- 
## 🧪 1. Chạy Tự Động Toàn Bộ Test Case Bằng Pytest
Chạy lệnh pytest để kiểm tra độ chính xác của 9/9 test cases trong bài.

In [ ]:
!pytest tests/ -v

--- 
## 🔹 Task 1: `call_openai` (GPT-4o)
Kiểm tra hàm gọi OpenAI, đo latency và token usage.

In [ ]:
prompt = "Hãy giải thích ngắn gọn AI là gì trong 1 câu."

# Chạy giả lập (Mocking) hoặc dùng Live API nếu đã cấu hình OPENAI_API_KEY
with patch("openai.OpenAI") as MockOpenAI:
    mock_client = MagicMock()
    MockOpenAI.return_value = mock_client
    mock_resp = MagicMock()
    mock_resp.choices = [MagicMock(message=MagicMock(content="Trí tuệ nhân tạo (AI) là khả năng máy tính mô phỏng tư duy của con người."))]
    mock_resp.usage = MagicMock(prompt_tokens=14, completion_tokens=25)
    mock_client.chat.completions.create.return_value = mock_resp

    text, latency, usage = call_openai(prompt)

print(f"[PROMPT]: {prompt}")
print(f"[OUTPUT]: {text}")
print(f"[LATENCY]: {latency:.4f}s")
print(f"[USAGE]: Input = {usage['input_tokens']} tokens | Output = {usage['output_tokens']} tokens")

--- 
## 🔹 Task 2: `call_gemini` (Gemini 2.5 Flash - Google GenAI SDK)
Kiểm tra gọi mô hình Gemini với `GenerateContentConfig` và trích xuất `usage_metadata`.

In [ ]:
prompt = "Vì sao Gemini 2.5 Flash có tốc độ phản hồi nhanh?"

with patch("google.genai.Client") as MockGenAI:
    mock_client = MagicMock()
    MockGenAI.return_value = mock_client
    mock_resp = MagicMock()
    mock_resp.text = "Gemini 2.5 Flash được tối ưu hóa kiến trúc để cung cấp độ trễ thấp và tối ưu chi phí suy luận."
    mock_resp.usage_metadata = MagicMock(prompt_token_count=16, candidates_token_count=30)
    mock_client.models.generate_content.return_value = mock_resp

    text, latency, usage = call_gemini(prompt)

print(f"[PROMPT]: {prompt}")
print(f"[OUTPUT]: {text}")
print(f"[LATENCY]: {latency:.4f}s")
print(f"[USAGE]: Input = {usage['input_tokens']} tokens | Output = {usage['output_tokens']} tokens")

--- 
## 🔹 Task 3: `call_anthropic` (Claude 3.5 Haiku)
Kiểm tra gọi mô hình Claude qua SDK Anthropic.

In [ ]:
prompt = "Thế mạnh của Claude 3.5 Haiku là gì?"

with patch("anthropic.Anthropic") as MockAnthropic:
    mock_client = MagicMock()
    MockAnthropic.return_value = mock_client
    mock_resp = MagicMock()
    mock_resp.content = [MagicMock(text="Claude 3.5 Haiku là mô hình cực kỳ nhanh, tiết kiệm và thông minh cho các tác vụ tác nhân.")]
    mock_resp.usage = MagicMock(input_tokens=15, output_tokens=28)
    mock_client.messages.create.return_value = mock_resp

    text, latency, usage = call_anthropic(prompt)

print(f"[PROMPT]: {prompt}")
print(f"[OUTPUT]: {text}")
print(f"[LATENCY]: {latency:.4f}s")
print(f"[USAGE]: Input = {usage['input_tokens']} tokens | Output = {usage['output_tokens']} tokens")

--- 
## 🔹 Task 4: `compare_models` (So sánh GPT-4o, GPT-4o-Mini, Gemini 2.5 Flash & Chi phí USD)

In [ ]:
prompt = "Phân biệt Supervised Learning và Unsupervised Learning."

with patch("solution.solution.call_openai") as mock_oa, \
     patch("solution.solution.call_gemini") as mock_gem:

    mock_oa.side_effect = [
        ("Supervised dùng dữ liệu có nhãn, Unsupervised dùng dữ liệu không nhãn (GPT-4o).", 0.65, {"input_tokens": 10, "output_tokens": 20}),
        ("Supervised có nhãn; Unsupervised không nhãn (GPT-4o-Mini).", 0.32, {"input_tokens": 10, "output_tokens": 20}),
    ]
    mock_gem.return_value = ("Học có giám sát dùng nhãn, không giám sát tự gom cụm (Gemini Flash).", 0.22, {"input_tokens": 10, "output_tokens": 20})

    comparison = compare_models(prompt)

import pandas as pd
df_data = []
for model_key, stats in comparison.items():
    df_data.append({
        "Model Key": model_key,
        "Response": stats["response"],
        "Latency (s)": f"{stats['latency']:.2f}",
        "Input Tokens": stats["input_tokens"],
        "Output Tokens": stats["output_tokens"],
        "Estimated Cost ($USD)": f"${stats['cost']:.8f}"
    })

pd.DataFrame(df_data)

--- 
## 🔹 Bonus Task A: `retry_with_backoff` (Xử lý Retry Lỗi và Exponential Backoff)

In [ ]:
call_counter = 0

def flaky_api_call():
    global call_counter
    call_counter += 1
    print(f"-> Đang thực hiện thử lần #{call_counter}...")
    if call_counter < 3:
        raise ConnectionResetError("Lỗi 429: Quá tải hệ thống (Rate Limit)!")
    return "Thành công sau khi retry!"

result = retry_with_backoff(flaky_api_call, max_retries=3, base_delay=0.1)
print(f"\n[KẾT QUẢ CUỐI CÙNG]: {result}")
print(f"[TỔNG SỐ LẦN ĐÃ THỰC THI]: {call_counter}")

--- 
## 🔹 Bonus Task B & C: `batch_compare` và `format_comparison_table` (Bảng Markdown)

In [ ]:
from IPython.display import Markdown, display

test_prompts = [
    "Python là gì?",
    "Deep Learning khác gì Machine Learning?"
]

with patch("solution.solution.compare_models") as mock_cm:
    def fake_compare(p):
        return {
            "gpt4o": {"response": f"GPT-4o: câu trả lời cho '{p}'", "latency": 0.52, "cost": 0.00045, "input_tokens": 10, "output_tokens": 20},
            "gpt4o_mini": {"response": f"Mini: câu trả lời cho '{p}'", "latency": 0.28, "cost": 0.0000135, "input_tokens": 10, "output_tokens": 20},
            "gemini_flash": {"response": f"Gemini: câu trả lời cho '{p}'", "latency": 0.21, "cost": 0.00000675, "input_tokens": 10, "output_tokens": 20},
        }
    mock_cm.side_effect = fake_compare

    batch_results = batch_compare(test_prompts)
    markdown_table = format_comparison_table(batch_results)

print("=== RAW MARKDOWN STRING ===\n")
print(markdown_table)

print("\n=== RENDER TRỰC QUAN GIAO DIỆN BẢNG ===\n")
display(Markdown(markdown_table))